# Classification EEG avec EEGNet

Ce notebook présente une approche de classification supervisée par apprentissage profond appliquée à des données EEG prétraitées. 

Le modèle utilisé est **EEGNet**, une architecture de réseau de neurones convolutionnel spécialement conçue pour l’analyse de signaux EEG.

À partir des époques EEG générées dans les étapes précédentes du pipeline, ce notebook réalise :
- le clonage du dépôt GitHub du projet dans Google Colab ;
- l’importation du modèle `EEGNet` depuis le fichier `EEGModels.py` ;
- le chargement des données EEG et des étiquettes depuis le dossier `output_data/` ;
- la préparation des tenseurs d’entrée pour le réseau ;
- la séparation des données en ensembles d’entraînement et de validation ;
- l’entraînement d’un modèle de base ;
- une exploration de plusieurs hyperparamètres ;
- l’évaluation des performances du classifieur ;
- la visualisation des résultats.

L’objectif est d’évaluer la capacité d’EEGNet à distinguer les deux classes à partir des signaux EEG, tout en gardant une structure de code compatible avec Google Colab et le dépôt GitHub du projet.

## 1. Clonage du dépôt GitHub et importation des bibliothèques

Cette section clone le dépôt GitHub du projet dans l’environnement Google Colab, puis importe les bibliothèques nécessaires à l’analyse. Le fichier `EEGModels.py`, maintenant stocké dans le dossier `/code` du dépôt, est ajouté au chemin Python afin de permettre l’importation du modèle EEGNet directement depuis le repo.

In [ ]:
# Cloner le dépôt GitHub dans l'environnement Colab
!git clone https://github.com/psy3019-6973-2026/Vallee_ProjetFinal.git

Pour pouvoir répliquer ce notebook, vous devez :

1. Avoir reproduit le pipeline localement avec `invoke fetch` et `invoke run`
2. Aller sur [drive.google.com](https://drive.google.com)
3. Créer un dossier `output_data` dans votre Google Drive
4. Y uploader les 4 fichiers `.npy` générés dans votre dossier `output_data/` local
5. Ensuite vous pourrez rouler les lignes suivantes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

drive_path = "/content/drive/MyDrive/output_data"
output_path = "/content/Vallee_ProjetFinal/output_data"

for fname in ["X_epochs_data.npy", "y_epoch_labels.npy", "X_features.npy"]:
    shutil.copy(os.path.join(drive_path, fname), os.path.join(output_path, fname))
    print(f"✅ {fname} copié")

In [ ]:
# Se déplacer à la racine du projet
%cd /content/Vallee_ProjetFinal

# Importations générales
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Définition des chemins du projet
projet_root = "/content/Vallee_ProjetFinal"
code_dir = os.path.join(projet_root, "code")
source_data_dir = os.path.join(projet_root, "source_data")
output_data_dir = os.path.join(projet_root, "output_data")

# Ajouter le dossier /code au chemin Python
sys.path.append(code_dir)

# Importer EEGNet depuis EEGModels.py
from EEGModels import EEGNet

## 2. Vérification de la présence des fichiers générés

Les fichiers nécessaires à l’entraînement du modèle (`X_epochs_data.npy`, `y_epoch_labels.npy` et `X_features.npy`) doivent avoir été générés par les étapes précédentes du pipeline et enregistrés dans le dossier `output_data/`. Cette cellule vérifie leur présence avant de poursuivre l’analyse.

In [ ]:
required_files = [
    "X_epochs_data.npy",
    "y_epoch_labels.npy",
    "X_features.npy",
    "subject_ids.npy"
]

for fname in required_files:
    fpath = os.path.join(output_data_dir, fname)
    print(f"{fname}: {'trouvé' if os.path.exists(fpath) else 'absent'}")

## 3. Chargement des données EEG prétraitées et des étiquettes

Cette section charge les époques EEG, les étiquettes de classe, les caractéristiques de bandpower et les identifiants de sujets sauvegardés dans le dossier `output_data/`. Ces fichiers ont été générés par le notebook de prétraitement et couvrent les trois conditions expérimentales (EC, EO et Task). Les dimensions des tableaux sont affichées pour confirmer que les données ont été correctement importées.

In [ ]:
# Chargement des données
X_epochs = np.load(os.path.join(output_data_dir, "X_epochs_data.npy"))
X_features = np.load(os.path.join(output_data_dir, "X_features.npy"))
y_labels = np.load(os.path.join(output_data_dir, "y_epoch_labels.npy"))
subject_ids = np.load(os.path.join(output_data_dir, "subject_ids.npy"))

# Vérification des dimensions
print("X_epochs shape :", X_epochs.shape)
print("X_features shape :", X_features.shape)
print("y_labels shape :", y_labels.shape)
print("subject_ids shape :", subject_ids.shape)

## 4. Préparation des tenseurs d’entrée pour EEGNet

EEGNet attend une entrée sous forme de tenseur 4D. Les époques EEG sont donc redimensionnées pour ajouter une dimension correspondant au canal d’entrée du réseau convolutionnel. Les étiquettes sont ensuite converties en format catégoriel (`one-hot encoding`) afin d’être compatibles avec la fonction de coût utilisée lors de l’entraînement.

In [ ]:
# Reshape des données EEG pour EEGNet : (n_epochs, n_channels, n_times, 1)
X_eeg = X_epochs.reshape(X_epochs.shape[0], X_epochs.shape[1], X_epochs.shape[2], 1)

# Encodage one-hot des labels
y_onehot = to_categorical(y_labels)

print("X_eeg shape :", X_eeg.shape)
print("y_onehot shape :", y_onehot.shape)

## 5. Séparation des données en ensembles d’entraînement et de validation

Les données sont divisées en deux sous-ensembles : un ensemble d'entraînement utilisé pour ajuster les poids du réseau, et un ensemble de test utilisé pour évaluer la performance finale du modèle.

La séparation est effectuée **au niveau des sujets** à l'aide de `GroupShuffleSplit`, en utilisant les identifiants de sujets (`subject_ids`). Cette approche garantit qu'aucune époque d'un même sujet ne se retrouve simultanément dans l'entraînement et dans le test, évitant ainsi tout **data leakage**. Sans cette précaution, le modèle pourrait apprendre à reconnaître les caractéristiques propres à chaque individu plutôt que les patterns généralisables liés au diagnostic.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_eeg, y_labels, groups=subject_ids))

X_train, X_val = X_eeg[train_idx],     X_eeg[test_idx]
y_train, y_val = y_onehot[train_idx],  y_onehot[test_idx]
y_true = np.argmax(y_val, axis=1)  # pour les matrices de confusion

print(f"Train : {len(X_train)} époques | Validation : {len(X_val)} époques")

Train size: 2584
Validation size: 1724


## 6. Définition d’une fonction utilitaire pour créer le modèle

Afin d’éviter de répéter les mêmes lignes de code à plusieurs endroits dans le notebook, une fonction utilitaire est définie pour construire et compiler un modèle EEGNet. Cette fonction permet d’ajuster facilement certains hyperparamètres tout en gardant une structure claire et réutilisable.

In [ ]:
def make_eegnet_model(
    nb_classes=2,
    chans=None,
    samples=None,
    dropout_rate=0.25,
    kern_length=64,
    F1=8,
    D=4,
    F2=32,
    learning_rate=0.001
):
    model = EEGNet(
        nb_classes=nb_classes,
        Chans=chans,
        Samples=samples,
        dropoutRate=dropout_rate,
        kernLength=kern_length,
        F1=F1,
        D=D,
        F2=F2,
        norm_rate=0.25,
        dropoutType='Dropout'
    )

    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=learning_rate),
        metrics=['accuracy']
    )
    return model

## 7. Entraînement du modèle de base

Un premier modèle EEGNet est ensuite entraîné avec une configuration initiale d’hyperparamètres. Un mécanisme d’**early stopping** est utilisé afin d’interrompre l’apprentissage lorsque la performance sur l’ensemble de validation cesse de s’améliorer, ce qui aide à limiter le surapprentissage.

In [ ]:
base_model = make_eegnet_model(
    nb_classes=2,
    chans=X_eeg.shape[1],
    samples=X_eeg.shape[2],
    dropout_rate=0.25,
    kern_length=64,
    F1=8,
    D=4,
    F2=32,
    learning_rate=0.001
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

base_history = base_model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=100,
    verbose=1,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

Epoch 1/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 12s 69ms/step - accuracy: 0.4778 - loss: 0.6932 - val_accuracy: 0.5070 - val_loss: 0.6931
Epoch 2/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.5631 - loss: 0.6900 - val_accuracy: 0.5070 - val_loss: 0.6931
Epoch 3/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6419 - loss: 0.6458 - val_accuracy: 0.5070 - val_loss: 0.7852
Epoch 4/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8069 - loss: 0.4590 - val_accuracy: 0.5070 - val_loss: 1.0339
Epoch 5/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8520 - loss: 0.3768 - val_accuracy: 0.5081 - val_loss: 1.3776
Epoch 6/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8685 - loss: 0.3470 - val_accuracy: 0.4930 - val_loss: 0.7627
Epoch 7/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8661 - loss: 0.3376 - val_accuracy: 0.4930 - val_loss: 2.9237
Epoch 8/100
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8760 - loss: 0.3277 - val_accuracy: 0

## 8. Évaluation du modèle de base

Les performances du modèle de base sont évaluées sur l’ensemble de validation. En plus de la perte et de l’accuracy, une matrice de confusion est générée afin d’examiner la répartition des prédictions correctes et incorrectes dans chaque classe.

In [ ]:
base_score = base_model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss : {base_score[0]:.4f}")
print(f"Validation accuracy : {base_score[1]:.4f}")

# Prédictions
y_pred_base = np.argmax(base_model.predict(X_val), axis=1)
y_true = np.argmax(y_val, axis=1)

# Matrice de confusion
cm_base = confusion_matrix(y_true, y_pred_base)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_base,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['H', 'MDD'],
    yticklabels=['H', 'MDD']
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Base EEGNet Model")
plt.show()

## 9. Visualisation de la courbe d’apprentissage du modèle de base

La courbe d’apprentissage permet de suivre l’évolution de l’accuracy sur les ensembles d’entraînement et de validation au fil des époques. Cette visualisation aide à juger si le modèle converge correctement et à repérer d’éventuels signes de surapprentissage.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(base_history.history['accuracy'], label='Train Acc')
plt.plot(base_history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('EEGNet Accuracy Curve - Base Model')
plt.grid()
plt.show()

## 10. Exploration des hyperparamètres

Après l’entraînement d’un modèle de base, une exploration des hyperparamètres est réalisée afin de tester si d’autres configurations d’EEGNet permettent d’améliorer les performances. Deux approches sont utilisées : une recherche automatisée avec **Keras-Tuner** et une comparaison manuelle de quelques combinaisons ciblées.

### 10.1 Recherche automatisée avec Keras-Tuner

Cette section utilise `keras-tuner` pour explorer automatiquement différentes combinaisons d’hyperparamètres. Chaque modèle testé est construit à partir de la fonction `build_model`, puis évalué selon son accuracy sur l’ensemble de validation.

In [ ]:
!pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 7.0 MB/s eta 0:00:00


Cette cellule définit la fonction utilisée par Keras-Tuner pour construire différentes variantes du modèle EEGNet.

In [ ]:
import keras_tuner as kt

def build_model(hp):
    model = make_eegnet_model(
        nb_classes=2,
        chans=X_eeg.shape[1],
        samples=X_eeg.shape[2],
        dropout_rate=hp.Float('dropout_rate', min_value=0.3, max_value=0.6, step=0.1),
        kern_length=hp.Choice('kern_length', [32, 64, 128]),
        F1=hp.Choice('F1', [2, 4, 6, 8]),
        D=hp.Choice('D', [1, 2, 4]),
        F2=hp.Choice('F2', [8, 16, 32]),
        learning_rate=hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
    )
    return model

Cette cellule lance la recherche aléatoire d’hyperparamètres et affiche la meilleure combinaison trouvée par Keras-Tuner.

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='eegnet_tuning',
    project_name='eegnet_classification'
)

tuner_early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

tuner.search(
    X_train,
    y_train,
    epochs=20,
    validation_data=(X_val, y_val),
    callbacks=[tuner_early_stop],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best hyperparameters found by Keras-Tuner:")
print(f"dropout_rate = {best_hps.get('dropout_rate')}")
print(f"kern_length = {best_hps.get('kern_length')}")
print(f"F1 = {best_hps.get('F1')}")
print(f"D = {best_hps.get('D')}")
print(f"F2 = {best_hps.get('F2')}")
print(f"learning_rate = {best_hps.get('learning_rate')}")

### 10.2 Recherche manuelle avec des boucles `for`

En complément de la recherche automatisée, quelques combinaisons ciblées d’hyperparamètres sont testées manuellement. Cette stratégie permet d’examiner rapidement l’effet de certaines configurations choisies sur la performance du modèle.

In [ ]:
manual_results = []
best_manual_acc = 0
best_manual_params = None
best_manual_model = None
best_manual_history = None

for F1 in [8, 16]:
    for D in [2, 4]:
        F2 = F1 * D

        current_model = make_eegnet_model(
            nb_classes=2,
            chans=X_eeg.shape[1],
            samples=X_eeg.shape[2],
            dropout_rate=0.25,
            kern_length=64,
            F1=F1,
            D=D,
            F2=F2,
            learning_rate=0.001
        )

        current_history = current_model.fit(
            X_train,
            y_train,
            batch_size=32,
            epochs=30,
            verbose=0,
            validation_data=(X_val, y_val)
        )

        current_val_acc = current_history.history['val_accuracy'][-1]

        manual_results.append({
            "F1": F1,
            "D": D,
            "F2": F2,
            "val_accuracy": current_val_acc
        })

        print(f"F1={F1}, D={D}, F2={F2} -> val_accuracy={current_val_acc:.4f}")

        if current_val_acc > best_manual_acc:
            best_manual_acc = current_val_acc
            best_manual_params = {"F1": F1, "D": D, "F2": F2}
            best_manual_model = current_model
            best_manual_history = current_history

print("\nBest manual parameters:", best_manual_params)
print(f"Best manual validation accuracy: {best_manual_acc:.4f}")

F1=8, D=2, val_acc=0.4930
F1=8, D=4, val_acc=0.4942
F1=16, D=2, val_acc=0.4942
F1=16, D=4, val_acc=0.8219
Best Params: {'F1': 16, 'D': 4}
Best Val Acc: 0.8219257593154907


## 11. Évaluation du meilleur modèle 

Le meilleur modèle issu de la recherche manuelle est ensuite évalué sur l’ensemble de validation. Cette séparation entre `base_model` et `best_manual_model` permet de savoir exactement quel modèle est en train d’être évalué, ce qui améliore la clarté du notebook.

In [ ]:
best_manual_score = best_manual_model.evaluate(X_val, y_val, verbose=0)
print(f"Validation loss (best manual model) : {best_manual_score[0]:.4f}")
print(f"Validation accuracy (best manual model) : {best_manual_score[1]:.4f}")

y_pred_best = np.argmax(best_manual_model.predict(X_val), axis=1)
cm_best = confusion_matrix(y_true, y_pred_best)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_best,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['H', 'MDD'],
    yticklabels=['H', 'MDD']
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Best Manual EEGNet Model")
plt.show()

## 12. Visualisation de la courbe d’apprentissage du meilleur modèle manuel

Cette dernière visualisation montre l’évolution de l’accuracy du meilleur modèle trouvé manuellement. Elle permet de comparer son comportement d’apprentissage avec celui du modèle de base.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(best_manual_history.history['accuracy'], label='Train Acc')
plt.plot(best_manual_history.history['val_accuracy'], label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('EEGNet Accuracy Curve - Best Manual Model')
plt.grid()
plt.show()

# Résumé du pipeline

Ce notebook comprend les étapes suivantes :

1. **Clonage du dépôt et importation des bibliothèques**
   Clonage du dépôt GitHub dans l'environnement Google Colab et importation des bibliothèques nécessaires à l'analyse. Le fichier `EEGModels.py` est importé depuis le dossier `/code` du dépôt.

2. **Vérification des fichiers générés**
   Vérification de la présence des fichiers `.npy` nécessaires dans le dossier `output_data/` avant de poursuivre l'analyse.

3. **Chargement des données EEG prétraitées et des étiquettes**
   Chargement des époques EEG, des caractéristiques de bandpower, des étiquettes de groupe et des identifiants de sujets générés par le notebook de prétraitement (conditions EC, EO et Task).

4. **Préparation des tenseurs d'entrée pour EEGNet**
   Reshape des époques EEG en tenseurs 4D compatibles avec l'architecture EEGNet, et encodage one-hot des étiquettes.

5. **Séparation des données en ensembles d'entraînement et de test**
   Division des données **au niveau des sujets** à l'aide de `GroupShuffleSplit` (via `subject_ids`), garantissant qu'aucun sujet ne se retrouve simultanément dans l'entraînement et dans le test. Cette approche évite le data leakage qui surviendrait lors d'un split aléatoire par époque.

6. **Définition d'une fonction utilitaire pour créer le modèle**
   Définition d'une fonction `make_eegnet_model()` permettant de construire et compiler une instance d'EEGNet avec des hyperparamètres ajustables.

7. **Entraînement du modèle de base**
   Entraînement d'un premier modèle EEGNet avec une configuration initiale d'hyperparamètres, en utilisant un mécanisme d'early stopping pour limiter le surapprentissage.

8. **Évaluation du modèle de base**
   Évaluation des performances sur l'ensemble de test, avec affichage de l'accuracy, de la matrice de confusion et du rapport de classification.

9. **Visualisation de la courbe d'apprentissage**
   Affichage de l'évolution de l'accuracy sur les ensembles d'entraînement et de validation au cours des époques d'entraînement.

10. **Recherche d'hyperparamètres**
    Exploration de différentes configurations d'hyperparamètres (taille des filtres, profondeur, taux de dropout) à l'aide d'une recherche automatisée (Keras Tuner) et d'une recherche manuelle par boucles.

11. **Évaluation du meilleur modèle**
    Évaluation du modèle le plus performant identifié lors de la recherche sur l'ensemble de test, avec affichage de la matrice de confusion finale.

12. **Visualisation de la courbe d'apprentissage du meilleur modèle**
    Comparaison de l'évolution de l'accuracy du meilleur modèle par rapport au modèle de base.